In [7]:
import os
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import ast


os.environ['GRB_LICENSE_FILE'] = r'C:\Users\anabe\gurobi.lic'


In [9]:
# 1. DADOS
pasta_dados =r'C:\Users\anabe\OneDrive\Área de Trabalho\Códigos\ALC\dados'

def ler_arquivo(nome_arquivo, tipo='lista'):
    
    caminho = os.path.join(pasta_dados, nome_arquivo)
    dados = {}
    
    # Verifica se o arquivo existe
    if not os.path.exists(caminho):
        print(f"ATENÇÃO: Arquivo não encontrado -> {caminho}")
        return dados
        
    with open(caminho, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if not linha: continue
            
            # Separa o ID da instância do resto do conteúdo (divide apenas na primeira vírgula)
            partes = linha.split(',', 1)
            id_inst = int(partes[0])
            resto = partes[1].strip()
            
            if tipo == 'lista':
                # Converte as strings "[...]" ou "[(...)]" em listas/tuplas reais do Python
                dados[id_inst] = ast.literal_eval(resto)
            elif tipo == 'parametro':
                # O número de alunos é o primeiro valor logo após o ID da instância
                num_alunos = int(resto.split(',')[0])
                dados[id_inst] = num_alunos
                
    return dados

dict_conflitos = ler_arquivo('grafo.txt', tipo='lista')
dict_frente = ler_arquivo('grafo_frente.txt', tipo='lista')
dict_atras = ler_arquivo('grafo_tras.txt', tipo='lista')
dict_J = ler_arquivo('grafo_proximidade.txt', tipo='lista')
dict_fileiras = ler_arquivo('grafo_vet_com_vazios.txt', tipo='lista')
dict_parametros = ler_arquivo('grafo_info.txt', tipo='parametro')
              

In [10]:
# 2. VARIAVEIS
for id_instancia in range(1, 31):
    print(f"{'='*50}")
    print(f" RESOLVENDO {id_instancia} ")
    print(f"{'='*50}")
    
    
    try:
        # 1. DADOS
        conflitos = dict_conflitos[id_instancia]
        frente = dict_frente[id_instancia]
        atras = dict_atras[id_instancia]
        J = dict_J[id_instancia]
        fileiras_cap = dict_fileiras[id_instancia]
        num_alunos = dict_parametros[id_instancia]
    except KeyError:
        print(f"Dados incompletos para a Instância {id_instancia}. Pulando para a próxima...\n")
        continue

    alunos = list(range(num_alunos))
    num_fileiras = len(fileiras_cap)
    
    #parametros
    P = 2*((len(conflitos) * max(fileiras_cap)) + 1)
    d_min = 2              

    # 2. VARIÁVEIS
    model = gp.Model(f"Instancia {id_instancia}")

    # x[i, L, k] é 1 se aluno i está na fileira L posição k
    x = {}
    for i in alunos:
        for L in range(num_fileiras):
            for k in range(fileiras_cap[L]):
                x[i, L, k] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{L}_{k}")

    # y[i, j] é 1 se a aresta de conflito entre i e j está ativa
    y = {}
    for (i, j) in conflitos:
        y[i, j] = model.addVar(vtype=GRB.BINARY, name=f"y_{i}_{j}")

    # w[L, i, j, k, z] lineariza o produto x[i,L,k] * x[j,L+1,z]
    w = {}
    for L in range(num_fileiras - 1):
        for (i, j) in conflitos:
            for k in range(fileiras_cap[L]):
                for z in range(fileiras_cap[L+1]):
                    w[L, i, j, k, z] = model.addVar(vtype=GRB.BINARY, name=f"w_{L}_{i}_{j}_{k}_{z}")
             
                    
    # 3. F.O.
    # max dist horizontal e min numero de conflitos
    obj = gp.quicksum(
        (abs(z - k) * w[L, i, j, k, z]) - (P * y[i, j])
        for L in range(num_fileiras - 1)
        for (i, j) in conflitos
        for k in range(fileiras_cap[L])
        for z in range(fileiras_cap[L+1])
    )
    model.setObjective(obj, GRB.MAXIMIZE)
    
    
    # 4. S.T.
    # cada aluno deve ocupar um assento
    for i in alunos:
        model.addConstr(gp.quicksum(x[i, L, k] for L in range(num_fileiras) for k in range(fileiras_cap[L])) == 1)

    # cada assento deve ter no máximo um aluno
    for L in range(num_fileiras):
        for k in range(fileiras_cap[L]):
            model.addConstr(gp.quicksum(x[i, L, k] for i in alunos) <= 1)
                    
    # se i e j em conflito estao na mesma fileira a dist >= d_min
    for (i, j) in conflitos:
        for L in range(num_fileiras):
            for k in range(fileiras_cap[L]):
                for z in range(k + 1, fileiras_cap[L]):
                    # restringe caso o aluno i sente a esquerda do aluno j
                    model.addConstr(z - k >= (x[i, L, k] + x[j, L, z] - 1) * d_min)
                    model.addConstr(z - k >= (x[j, L, k] + x[i, L, z] - 1) * d_min)                

    # definição de aresta ativa (y)
    for (i, j) in conflitos:
        for L in range(num_fileiras - 1):
            for k in range(fileiras_cap[L]):
                for z in range(fileiras_cap[L+1]):
                    # i em L,k e j em L+1,z -> aresta fica ativa
                    model.addConstr(y[i, j] >= x[i, L, k] + x[j, L+1, z] - 1)
                    model.addConstr(y[i, j] >= x[j, L, k] + x[i, L+1, z] - 1)

                    # def W
                    model.addConstr(w[L, i, j, k, z] <= x[i, L, k])
                    model.addConstr(w[L, i, j, k, z] <= x[j, L+1, z])
                    model.addConstr(w[L, i, j, k, z] >= x[i, L, k] + x[j, L+1, z] - 1)

    # alunos na lista frente: k=1 ou k=2
    for i_frente in frente:
        model.addConstr(gp.quicksum(x[i_frente, L, k] for L in range(num_fileiras) for k in [0, 1]) == 1)

    # alunos na lista atras: duas últimas posições
    for i_atras in atras:
        model.addConstr(gp.quicksum(x[i_atras, L, k] for L in range(num_fileiras) for k in [fileiras_cap[L]-2, fileiras_cap[L]-1]) == 1)
        
    # NOVA RESTRIÇÃO: proximidade obrigatória (S)
    for (i, j) in J:
        for L in range(num_fileiras):
            for k in range(fileiras_cap[L]):
                vizinhos = []
                
                # Lado
                if k > 0: 
                    vizinhos.append(x[j, L, k-1]) # Esquerda
                if k < fileiras_cap[L] - 1: 
                    vizinhos.append(x[j, L, k+1]) # Direita
                    
                # Frente e Trás
                if L > 0 and k < fileiras_cap[L-1]: 
                    vizinhos.append(x[j, L-1, k]) # Frente
                if L < num_fileiras - 1 and k < fileiras_cap[L+1]: 
                    vizinhos.append(x[j, L+1, k]) # Atrás
                
                # Se o aluno i sentar em (L, k), o aluno j DEVE estar em um dos vizinhos
                model.addConstr(x[i, L, k] <= gp.quicksum(vizinhos), name=f"dupla_{i}_{j}_{L}_{k}")
                
               
               
                
    # 5. RESULTADO
    model.Params.TimeLimit = 300
    model.optimize()

    status_map = {
            GRB.OPTIMAL: "SOLUÇÃO ÓTIMA",
            GRB.TIME_LIMIT: "LIMITE DE TEMPO ATINGIDO",
            GRB.INFEASIBLE: "MODELO INVIÁVEL",
            GRB.INF_OR_UNBD: "INVIÁVEL OU ILIMITADO"
    }
    status_msg = status_map.get(model.Status, f"DESCONHECIDO")

    print(f"\nINSTÂNCIA {id_instancia:02d} | Status: {model.Status} ({status_msg})")
    print(f"Tempo de Execução: {model.Runtime:.2f} s")

    if model.SolCount > 0:
        print(f"Valor da FO: {model.ObjVal:.2f}")
        print(f"MIPGap: {model.MIPGap * 100:.4f}%")
        
        # 1. construcao e impressao do mapa visual da sala
        print("\n" + "="*25 + " MAPA DA SALA DE AULA " + "="*25)
        
        # cria uma estrutura vazia para representar a sala baseada nas capacidades
        mapa_sala = []
        for l in range(num_fileiras):
            # preenche inicialmente cada assento como vazio
            mapa_sala.append(["[====]"] * fileiras_cap[l]) 
            
        # mapeia os alunos encontrados pelo gurobi para suas respectivas coordenadas
        posicao_alunos = {}
        for i in alunos:
            for l in range(num_fileiras):
                for k in range(fileiras_cap[l]):
                    if x[i, l, k].X > 0.5:
                        posicao_alunos[i] = np.array([l, k])
                        mapa_sala[l][k] = f"[Al.{i:02d}]"

        # imprime o mapa da sala linha por linha (da frente para o fundo)
        print(" [FRENTE DA SALA / QUADRO] ".center(68, "-"))
        for l in range(num_fileiras):
            linha_str = f"Fileira {l}: "
            for k in range(fileiras_cap[l]):
                linha_str += mapa_sala[l][k] + " "
            print(linha_str)
        print("-" * 68)
        print("Legenda: [Al.XX] = Aluno Alocado | [====] = Carteira Vazia")
        
        # 2. memorial de calculo detalhado das normas para o relatorio
        print("\n" + "="*20 + " MEMORIAL DE CÁLCULO DAS NORMAS " + "="*20)
        
        for (aluno_a, aluno_b) in J:
            if aluno_a in posicao_alunos and aluno_b in posicao_alunos:
                vec_a = posicao_alunos[aluno_a]
                vec_b = posicao_alunos[aluno_b]
                
                # calculo do vetor diferenca (passo 1)
                vec_diff = vec_a - vec_b
                
                # calculo explicito da norma l1 (passo 2)
                l1_passo = abs(vec_diff[0]) + abs(vec_diff[1])
                norma_l1 = np.linalg.norm(vec_diff, ord=1)
                
                # calculo explicito da norma l2 (passo 3)
                l2_passo = (vec_diff[0]**2) + (vec_diff[1]**2)
                norma_l2 = np.linalg.norm(vec_diff, ord=2)
                
                # calculo explicito da norma l-infinito (passo 4)
                linf_passo = max(abs(vec_diff[0]), abs(vec_diff[1]))
                norma_linf = np.linalg.norm(vec_diff, ord=np.inf)
                
                print(f"\nAnálise Geométrica do Par Estratégico: Aluno {aluno_a} e Aluno {aluno_b:=2d}")
                print(f"  -> Vetor Posição Aluno {aluno_a}: v_{aluno_a} = [{vec_a[0]}, {vec_a[1]}]")
                print(f"  -> Vetor Posição Aluno {aluno_b}: v_{aluno_b} = [{vec_b[0]}, {vec_b[1]}]")
                print(f"  -> Vetor Diferença (d): v_{aluno_a} - v_{aluno_b} = [{vec_diff[0]}, {vec_diff[1]}]")
                
                print(f"  * Desenvolvimento da Norma L1 (Manhattan):")
                print(f"    ||d||_1 = |{vec_diff[0]}| + |{vec_diff[1]}| = {l1_passo} -> (Distância por corredores ortogonais)")
                
                print(f"  * Desenvolvimento da Norma L2 (Euclidiana):")
                print(f"    ||d||_2 = sqrt({vec_diff[0]}^2 + {vec_diff[1]}^2) = sqrt({l2_passo}) = {norma_l2:.2f} -> (Distância em linha reta)")
                
                print(f"  * Desenvolvimento da Norma L-infinito (Máximo):")
                print(f"    ||d||_\u221e = max(|{vec_diff[0]}|, |{vec_diff[1]}|) = {linf_passo} -> (Raio da caixa delimitadora)")
                print("-" * 60)
                
        # analise dos pares em conflito
        print("\n" + "="*20 + " ESTATÍSTICA DOS CONFLITOS " + "="*20)
            
        
        # bloco de analise dos alunos em conflito        
        conflitos_l1 = []
        conflitos_l2 = []
        conflitos_linf = []
            
        # calcula as normas para cada par de alunos que nao podem ficar juntos
        for (aluno_a, aluno_b) in conflitos:
            if aluno_a in posicao_alunos and aluno_b in posicao_alunos:
                vetor_diff = posicao_alunos[aluno_a] - posicao_alunos[aluno_b]
                    
                conflitos_l1.append(np.linalg.norm(vetor_diff, ord=1))
                conflitos_l2.append(np.linalg.norm(vetor_diff, ord=2))
                conflitos_linf.append(np.linalg.norm(vetor_diff, ord=np.inf))
            
            # exibe as medias se houver conflitos alocados
            if conflitos_l2:
                media_l1 = np.mean(conflitos_l1)
                media_l2 = np.mean(conflitos_l2)
                media_linf = np.mean(conflitos_linf)
                
                menor_l1 = np.min(conflitos_l1)
                menor_l2 = np.min(conflitos_l2)
                menor_linf = np.min(conflitos_linf)
                
        print(f"quantidade de pares em conflito avaliados: {len(conflitos_l2)}")
                
        print(f"\n--- distancias medias ---")
        print(f"media l1 (corredores): {media_l1:.2f} carteiras")
        print(f"media l2 (linha reta): {media_l2:.2f} unidades")
        print(f"media l-infinito (maior eixo): {media_linf:.2f}")       
                
        print(f"\n--- distancias minimas (pior caso) ---")
        print(f"menor l1 (corredores): {menor_l1} carteiras")
        print(f"menor l2 (linha reta): {menor_l2:.2f} unidades")
        print(f"menor l-infinito (maior eixo): {menor_linf}")   
                
    else:
        print("Valor da FO: N/A (Inviável)")
        print("RESULTADO: Sem solução.")

 RESOLVENDO 1 
Set parameter TimeLimit to value 300
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-1035G1 CPU @ 1.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 23308 rows, 5037 columns and 60916 nonzeros
Model fingerprint: 0x5eb92e97
Variable types: 0 continuous, 5037 integer (5037 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [1e+00, 5e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]
Found heuristic solution: objective -234717.0000
Presolve removed 15394 rows and 2500 columns
Presolve time: 0.30s
Presolved: 7914 rows, 2537 columns, 22439 nonzeros
Variable types: 0 continuous, 2537 integer (2537 binary)

Root relaxation: objective 1.381544e+02, 5393 iterations, 0.28 seconds (0.28 work units)

    Nodes    |    Current N